# YOLO line pre-fill for 1951_01

Trains a line detector on your existing Label Studio annotations, then pre-fills boxes and ground-truth text for 1951_01.
Runtime: **GPU** (Runtime > Change runtime type > T4). Every cell is safe to re-run.

Before running:
1. Put `yolo_prefill.py` in `jawi_dataset/` on Drive (next to `generate_tasks.py`, `sheet_discovery_fix.py`).
2. Put your latest Label Studio export JSON in `jawi_dataset/exports/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/jawi_dataset/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/jawi_dataset


In [ ]:
!pip install -q ultralytics pymupdf gspread

In [ ]:
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
from googleapiclient.discovery import build
creds, _ = default()
gc = gspread.authorize(creds)
drive_service = build('drive', 'v3', credentials=creds)

In [ ]:
import sys, json, inspect
from pathlib import Path
for m in ['yolo_prefill', 'generate_tasks', 'sheet_discovery_fix', 'prepare_jawi_dataset_v2', 'ground_truth_gsheets']:
    sys.modules.pop(m, None)
sys.path.insert(0, '/content/drive/MyDrive/jawi_dataset')
import yolo_prefill as yp
import generate_tasks, sheet_discovery_fix as sdf
import prepare_jawi_dataset_v2 as tool
import ground_truth_gsheets as ggt
assert 'max_dimension' in inspect.getsource(generate_tasks.image_to_data_url), 'OLD generate_tasks.py on Drive'
print(yp.__file__)

/content/drive/MyDrive/jawi_dataset/yolo_prefill.py


## 1. Config

In [ ]:
EXPORT_JSON = '/content/drive/MyDrive/jawi_dataset/exports/project-2-at-2026-10-23-20-34-05ec7cdf.json'  # <- your latest export

# Locked test articles: never used to train the detector or TrOCR. Change only BEFORE any training.
TEST_ARTICLES = ['1950_11_009', '1950_11_013', '1950_12_003', '1950_12_007', '1950_12_010', '1950_12_014']

# Label Studio task 1 has no article_id/page_index (1950_11_002 page 0).
OVERRIDES = {1: ('1950_11_002', 0)}

YOLO_DS = '/content/yolo_ds'                                     # local disk: faster than Drive
RUNS = '/content/drive/MyDrive/jawi_dataset/yolo_runs'           # weights saved to Drive
BEST = Path(RUNS) / 'qalam_lines' / 'weights' / 'best.pt'

NEW_ISSUE = '1951_01'
PDF_DIR = Path('/content/drive/MyDrive/Al_Qalam/Al_Qalam_Jawi_Full')
CSV_FOLDER_IDS = {'1951_01': '1oBYEN2GNAJwOGvh28dmeZAO4JlxbesRr'}

## 2. Build the YOLO dataset from your annotations

In [ ]:
summary = yp.export_to_yolo(EXPORT_JSON, YOLO_DS, exclude_articles=TEST_ARTICLES, overrides=OVERRIDES)
print(json.dumps({k: v for k, v in summary.items()}, indent=1, default=str))
# Expect: train ~57 pages / ~3,250 boxes, val ~6 pages. 'incomplete_pages' are pages with boxes but no text: finish them in Label Studio.

{
 "train_pages": 58,
 "train_boxes": 3317,
 "val_pages": 6,
 "val_boxes": 366,
 "val_articles": [
  "1950_11_015",
  "1950_12_005",
  "1950_12_008"
 ],
 "excluded_test_articles": [
  "1950_11_009",
  "1950_11_013",
  "1950_12_003",
  "1950_12_007",
  "1950_12_010",
  "1950_12_014"
 ],
 "no_metadata": [],
 "duplicates_dropped": [],
 "incomplete_pages": [],
 "no_image": []
}


## 3. Train (about 15-25 min on a T4). Skips if weights already exist.

In [ ]:
import torch
print('GPU:', torch.cuda.is_available())
if BEST.exists():
    print('Using existing', BEST)
else:
    best = yp.train(f'{YOLO_DS}/data.yaml', project=RUNS, name='qalam_lines', epochs=80, imgsz=1280, batch=8, patience=20)
    print('Saved', best)
    BEST = Path(best)

GPU: False
Using existing /content/drive/MyDrive/jawi_dataset/yolo_runs/qalam_lines/weights/best.pt


## 4. Honest check on the locked test articles (never trained on)

In [ ]:
from ultralytics import YOLO
import cv2
model = YOLO(str(BEST))
pages, _ = yp.load_pages(EXPORT_JSON, OVERRIDES)
rows = []
for p in pages:
    if p['article_id'] not in TEST_ARTICLES or p['incomplete']:
        continue
    img = yp._decode_data_url(p['image_url']); h, w = img.shape[:2]
    gold = [(x/100*w, y/100*h, (x+bw)/100*w, (y+bh)/100*h) for (x, y, bw, bh) in p['boxes']]
    pred = yp.predict_boxes(model, img, imgsz=960)
    s = yp.match_stats(pred, gold, thr=0.5)
    rows.append((p['article_id'], p['page_index'], s['pred'], s['gold'], s['precision'], s['recall']))
for r in rows: print(*r)
tp_p = sum(r[4]*r[2] for r in rows); tp_r = sum(r[5]*r[3] for r in rows)
print(f"\nOverall precision {tp_p/sum(r[2] for r in rows):.3f}  recall {tp_r/sum(r[3] for r in rows):.3f}  (IoU>=0.5)")
# Rough guide: recall >= 0.85 means most boxes will already be right when you open Label Studio.

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
1950_11_009 0 46 49 0.913 0.857
1950_11_009 1 54 63 1.0 0.857
1950_11_009 2 56 61 1.0 0.918
1950_11_013 0 15 17 0.667 0.588
1950_12_003 0 63 61 0.905 0.934
1950_12_003 1 67 65 0.94 0.969
1950_12_007 0 70 83 0.886 0.747
1950_12_010 0 58 64 0.966 0.875
1950_12_010 1 43 47 0.93 0.851
1950_12_010 2 43 44 0.953 0.932
1950_12_010 3 52 57 1.0 0.912
1950_12_014 0 23 23 0.826 0.826
1950_12_014 1 13 7 0.538 1.0

Overall precision 0.927  recall 0.872  (IoU>=0.5)


## 5. Find 1951_01 articles (scans + corrected sheets)

In [ ]:
articles = tool.discover_articles_gsheets(PDF_DIR, drive_service, CSV_FOLDER_IDS, issue_prefixes=(NEW_ISSUE,))
sdf.attach_sheet_ids(articles, drive_service, CSV_FOLDER_IDS)
ready = {a: i for a, i in articles.items() if i.get('sheet_id') and i.get('pdf') and 'DONE' in (i.get('sheet_name') or '').upper()}
print(len(ready), 'ready:', sorted(ready))
print('NOT READY (no scan, no sheet, or sheet not _DONE):', sorted(set(articles) - set(ready)))

No sheet found for: ['1951_01_019']
16 ready: ['1951_01_002', '1951_01_003', '1951_01_005', '1951_01_006', '1951_01_009', '1951_01_011', '1951_01_015', '1951_01_021', '1951_01_022', '1951_01_024', '1951_01_029', '1951_01_032', '1951_01_034', '1951_01_035', '1951_01_040', '1951_01_043']
NOT READY (no scan, no sheet, or sheet not _DONE): ['1951_01_001', '1951_01_019', '1951_01_044']


## 6. Pre-fill Label Studio tasks with detector boxes + ground-truth text

In [ ]:
summary_rows, failed = [], []
for article_id in sorted(ready):
    out_json = Path('dataset') / article_id / 'ls_tasks.json'
    if not sdf.needs_generation(out_json):
        print('skip', article_id); continue
    info = ready[article_id]
    try:
        tool.render_article(info['pdf'], Path('dataset'), article_id)
        page_paths = sorted((Path('dataset') / article_id / 'pages').glob('page_*.png'))
        gt_lines = sdf.with_retry(ggt.load_article_lines_from_sheet, info['sheet_id'], gspread_client=gc)
        if not sdf.sanity_check_lines(article_id, gt_lines):
            failed.append((article_id, 'GT too short')); continue
        tasks, st = yp.prefill_article(model, page_paths, gt_lines, article_id, imgsz=960)
        generate_tasks.write_tasks_json(tasks, str(out_json))
        summary_rows.append((article_id, st['pages'], st['total_boxes'], st['gt_lines'], f"{out_json.stat().st_size/1e6:.1f}MB"))
    except Exception as e:
        print('FAILED', article_id, e); failed.append((article_id, str(e)[:120]))

print('\narticle | pages | detector boxes | GT lines | size')
for r in summary_rows:
    print(*r, '' if abs(r[2] - r[3]) <= 3 else '  <-- check')
print('FAILED:', failed)

  rendered page 0: dataset/1951_01_002/pages/page_00.png (3284x4678)
Wrote 1 tasks to dataset/1951_01_002/ls_tasks.json
  rendered page 0: dataset/1951_01_003/pages/page_00.png (3284x4678)
  rendered page 1: dataset/1951_01_003/pages/page_01.png (3284x4678)
  rendered page 2: dataset/1951_01_003/pages/page_02.png (3284x4678)
Wrote 3 tasks to dataset/1951_01_003/ls_tasks.json
  rendered page 0: dataset/1951_01_005/pages/page_00.png (3284x4678)
Wrote 1 tasks to dataset/1951_01_005/ls_tasks.json
  rendered page 0: dataset/1951_01_006/pages/page_00.png (3284x4678)
  rendered page 1: dataset/1951_01_006/pages/page_01.png (3284x4684)
  rendered page 2: dataset/1951_01_006/pages/page_02.png (3267x4678)
Wrote 3 tasks to dataset/1951_01_006/ls_tasks.json
  rendered page 0: dataset/1951_01_009/pages/page_00.png (3267x4678)
  rendered page 1: dataset/1951_01_009/pages/page_01.png (3267x4678)
Wrote 2 tasks to dataset/1951_01_009/ls_tasks.json
  rendered page 0: dataset/1951_01_011/pages/page_00.pn

## 7. Import into Label Studio
Import each `dataset/1951_01_XXX/ls_tasks.json` as before. Boxes and text are pre-filled; you correct them.
Note: text is assigned in reading order, so one missed or extra box shifts the text after it on that page. Fix the box, then drag text if needed.